# `geoai-datacubes` — Integration with `opengeos/geoai`

<a href="https://colab.research.google.com/github/buckai-observatory/geoai-datacubes/blob/main/notebooks/03_with_opengeos_geoai.ipynb" target="_blank" rel="noopener noreferrer"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


This notebook demonstrates how `geoai-datacubes` (data-prep front-end)
and `geoai-py` (Wu, 2026, JOSS 11(118):9605 — modelling back-end)
compose into a single workflow. We build fused multi-mission cubes
for three Ohio cities with our pipeline, then hand them to two `geoai-py`
entry points in turn:

1. **§2 Pretrained inference** — `geoai.segment_water` on a NAIP scene.
   One function call, no training, OmniWaterMask + OSM overlay.
2. **§3 Custom training** — fused cube → `select_bands` →
   `geoai.train_segmentation_landcover` → `geoai.semantic_segmentation`.
   Our `select_bands` helper writes the clean 3- or 4-band uint8 GeoTIFF
   that `geoai-py`'s loaders require; `geoai-py` handles the U-Net + loss
   + sliding-window inference.

**This notebook is deliberately honest about generalisation.** We train
on Cleveland + Cincinnati and **hold all of Columbus out** as an unseen
test region. In-distribution F1 reaches ~0.9; out-of-distribution F1 on
Columbus collapses to ~0.05. The integration works cleanly in both
directions; the OOD failure is the normal remote-sensing-ML story
(model overfits to land-cover patterns it saw) and the lesson is one
the user should see explicitly rather than be misled into thinking the
in-distribution number is the full picture.

In [ ]:
%%time
# --- Colab / local bootstrap ---
# On Colab, this cell clones the repo and installs the geoai-cubes deps
# (geoai-py, omniwatermask, torchgeo, leafmap, segmentation_models_pytorch).
# Expect ~2-3 min on a Colab cold start; near-instant locally.
import os, subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected -- bootstrapping repo + deps (this takes 2-3 min)")
    REPO_DIR = Path("/content/geoai-datacubes")
    if not REPO_DIR.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/buckai-observatory/geoai-datacubes.git",
            str(REPO_DIR),
        ])

    pip_pkgs = ["pystac", "pystac-client", "planetary-computer",
                "geoai-py", "omniwatermask", "torchgeo", "leafmap",
                "segmentation-models-pytorch"]
    missing = []
    for pkg in pip_pkgs:
        check = pkg.replace("-", "_").replace("geoai_py", "geoai")
        try:
            __import__(check)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"pip install -q {' '.join(missing)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                              *missing])

    # Editable install of the repo so we can import geoai_datacubes itself.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e",
                          str(REPO_DIR)])

    os.chdir(REPO_DIR / "notebooks")
    print(f"cwd = {os.getcwd()}")
else:
    print("Local environment -- using existing checkout")


## Setup

Import the pipeline + `geoai-py` and set up a scratch output folder.
The `_outputs_nb03/` directory under `notebooks/` will hold all fetched
imagery, fused cubes, training chips, model checkpoints, and prediction
rasters — so the rest of the repo stays clean and `rm -rf _outputs_nb03`
is a safe reset.

In [ ]:
import os, sys, time, shutil, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling

# Locate the repo root from the notebook's CWD.
NB_DIR = Path.cwd()
if NB_DIR.name != "notebooks":
    for p in (NB_DIR, *NB_DIR.parents):
        if (p / "notebooks").is_dir() and (p / "geoai_datacubes").is_dir():
            NB_DIR = p / "notebooks"
            break
REPO_ROOT = NB_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

# Our package: data-prep front-end
import geoai_datacubes
from geoai_datacubes.fetch import (
    resolve_aoi,
    fetch_sentinel_data,
    MISSION_PROFILES,
)
from geoai_datacubes.preprocessing import (
    fuse_response_tiffs,
    select_bands,
    write_label_uint8,
    BAND_PRESETS,
)
from geoai_datacubes.viz.scenes import find_response

# opengeos/geoai: modelling back-end
import geoai

# Scratch output dir.
OUT_DIR = NB_DIR / "_outputs_nb03"
OUT_DIR.mkdir(exist_ok=True)

# Cosmetic + reproducibility.
plt.rcParams["figure.dpi"] = 100
plt.rcParams["image.cmap"] = "viridis"
warnings.filterwarnings("ignore", category=UserWarning, module="rasterio")
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"geoai_datacubes v{geoai_datacubes.__version__}")
print(f"geoai           v{geoai.__version__}")
print(f"missions        : {len(MISSION_PROFILES)}")
print(f"BAND_PRESETS    : {list(BAND_PRESETS.keys())}")
print(f"output dir      : {OUT_DIR.resolve()}")


## 1. Build fused multi-mission cubes for three Ohio cities

We define a 3-mile box centered on the downtown of each of three Ohio
cities. The three deliberately span very different water morphologies,
which sets up the §3 cross-region experiment:

| City | Center | Water class | Type |
|---|---|---:|---|
| **Cleveland** | Public Square | ~25 % | Lake Erie shoreline (large continuous water) |
| **Cincinnati** | Fountain Square | ~10 % | Wide Ohio River |
| **Columbus** | Downtown | ~1 % | Narrow Olentangy + Scioto rivers |

For each city we fetch four optical / radar / DEM missions and one
land-cover label source, then fuse the first three onto a common 10 m
UTM grid:

| Mission | Resolution | Role |
|---|---|---|
| `NAIP` | 1 m | High-res RGB+NIR; input for `segment_water` in §2 |
| `Sentinel-2` | 10 m | Multispectral (B02, B03, B04, B08) |
| `Sentinel-1` | 10 m | SAR backscatter (VV, VH) |
| `Copernicus-DEM` | 30 m | Terrain |
| `ESA-WorldCover` | 10 m | LULC label → remapped to binary water mask |

In [ ]:
# ============================================================
# USER INPUT
# ============================================================
CITIES = {
    "cleveland":  (41.4993, -81.6944),   # Public Square, lake shore
    "cincinnati": (39.0997, -84.5147),   # Fountain Square, Ohio River
    "columbus":   (40.0067, -83.0305),   # downtown, narrow rivers
}
AOI_SIDE_MILES = 3

TIME_RANGE_OPTICAL = ("2024-06-15", "2024-07-15")
TIME_RANGE_RADAR   = ("2024-06-01", "2024-06-30")
TIME_RANGE_NAIP    = ("2022-01-01", "2024-12-31")

BANDS_NAIP = ["R", "G", "B", "NIR"]
BANDS_S2   = ["B02", "B03", "B04", "B08"]
BANDS_S1   = ["VV", "VH"]
BANDS_DEM  = ["DEM"]
BANDS_LULC = ["LULC"]

FUSE_RES = 10

for city, center in CITIES.items():
    bbox = resolve_aoi({"center": center, "side_miles": AOI_SIDE_MILES})
    print(f"{city:11s}  bbox=[{', '.join(f'{x:7.3f}' for x in bbox)}]")


In [ ]:
%%time
# Fetch each mission, fuse the feature cube, and build the binary water mask.
# Cold start: ~5-7 min total for the 3 cities (15 fetches at ~5-60s each
# + 3 fuses + 3 mask reprojections). Cached re-runs: a few seconds.

def build_city(city, center):
    bbox = resolve_aoi({"center": center, "side_miles": AOI_SIDE_MILES})
    city_dir = OUT_DIR / city
    data_dir = city_dir / "data";  data_dir.mkdir(parents=True, exist_ok=True)
    img_dir  = city_dir / "image"; img_dir.mkdir(exist_ok=True)
    msk_dir  = city_dir / "mask";  msk_dir.mkdir(exist_ok=True)

    missions = [
        ("NAIP",           BANDS_NAIP, 1,  TIME_RANGE_NAIP),
        ("Sentinel-2",     BANDS_S2,   10, TIME_RANGE_OPTICAL),
        ("Sentinel-1",     BANDS_S1,   10, TIME_RANGE_RADAR),
        ("Copernicus-DEM", BANDS_DEM,  30, TIME_RANGE_OPTICAL),
        ("ESA-WorldCover", BANDS_LULC, 10, TIME_RANGE_OPTICAL),
    ]

    scenes = {}
    for mission, bands, res, trange in missions:
        cached = sorted(data_dir.glob(f"{mission}_*"), key=os.path.getmtime)
        if cached and list(cached[-1].glob("*_full_size.tiff")):
            scenes[mission] = cached[-1]
            continue
        t0 = time.time()
        try:
            fetch_sentinel_data(
                mission, bands, trange, bbox,
                resolution=res, save_folder=str(data_dir),
                max_cloud_coverage=0.10, min_cloud_coverage=0.0,
                provider="auto",
            )
        except Exception as e:
            print(f"  {city}/{mission} FAIL: {type(e).__name__}: {str(e)[:80]}")
            continue
        new = sorted(data_dir.glob(f"{mission}_*"), key=os.path.getmtime)
        if new:
            scenes[mission] = new[-1]
            print(f"  {city}/{mission:18s} OK in {time.time()-t0:5.1f}s")

    cube_path = img_dir / "cube_10m.tif"
    if not cube_path.exists():
        fuse_response_tiffs(
            inputs=[find_response(scenes[m])
                    for m in ("Sentinel-2", "Sentinel-1", "Copernicus-DEM")
                    if m in scenes],
            output_path=str(cube_path),
            resolution=FUSE_RES, dst_crs=None, bbox_mode="intersection",
        )

    mask_path = msk_dir / "water_mask_10m.tif"
    if not mask_path.exists() and "ESA-WorldCover" in scenes:
        with rasterio.open(cube_path) as cube:
            meta = cube.meta.copy()
            ct, ccrs, H, W = cube.transform, cube.crs, cube.height, cube.width
        with rasterio.open(find_response(scenes["ESA-WorldCover"])) as wc:
            dst = np.zeros((H, W), dtype=np.uint8)
            reproject(
                source=wc.read(1), destination=dst,
                src_transform=wc.transform, src_crs=wc.crs,
                dst_transform=ct, dst_crs=ccrs,
                resampling=Resampling.nearest,
            )
        water = (dst == 80).astype(np.uint8)
        meta.update(count=1, dtype="uint8", nodata=None)
        with rasterio.open(mask_path, "w", **meta) as o:
            o.write(water, 1)

    naip_path = find_response(scenes["NAIP"]) if "NAIP" in scenes else None
    return dict(cube=cube_path, mask=mask_path, naip=naip_path)


CITY_FILES = {}
for city, center in CITIES.items():
    print(f"\n=== {city.upper()} ===")
    CITY_FILES[city] = build_city(city, center)

print("\n=== summary ===")
for city, files in CITY_FILES.items():
    with rasterio.open(files["cube"]) as c:
        with rasterio.open(files["mask"]) as m: water_pct = 100 * m.read(1).mean()
        print(f"  {city:11s}  cube={c.count}b x {c.height}x{c.width}  "
              f"water={water_pct:5.2f}%  NAIP={files['naip'].name if files['naip'] else 'NONE'}")


In [ ]:
# Visualise NAIP RGB, S2 RGB, S1 VV, DEM, and the water mask for each city.
def _stretch(arr, lo=2, hi=98):
    f = arr[np.isfinite(arr)]
    if f.size == 0:
        return np.zeros_like(arr, dtype=np.uint8)
    a, b = np.percentile(f, [lo, hi])
    return np.clip((arr - a) * 255 / max(b - a, 1e-9), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(len(CITIES), 5, figsize=(20, 4 * len(CITIES)))
for row, (city, files) in enumerate(CITY_FILES.items()):
    with rasterio.open(files["cube"]) as src:
        cube = src.read().astype("float32")
        descs = list(src.descriptions or [])
    s2_b, s2_g, s2_r = (descs.index(f"Sentinel-2_B0{n}") for n in (2, 3, 4))
    s1_vv = descs.index("Sentinel-1_VV")
    dem_i = descs.index("Copernicus-DEM_DEM")

    s2_rgb = np.dstack([_stretch(cube[s2_r]),
                        _stretch(cube[s2_g]),
                        _stretch(cube[s2_b])])
    with rasterio.open(files["naip"]) as n:
        naip_rgb = np.dstack([_stretch(n.read(i).astype("float32"))
                              for i in (1, 2, 3)])
    with rasterio.open(files["mask"]) as m:
        water = m.read(1)

    panels = [
        (naip_rgb,         f"{city.capitalize()} -- NAIP RGB"),
        (s2_rgb,           "Sentinel-2 RGB"),
        (cube[s1_vv],      "Sentinel-1 VV (gamma0)"),
        (cube[dem_i],      "Copernicus DEM (m)"),
        (water,            f"WorldCover water ({100*water.mean():.1f}%)"),
    ]
    for col, (img, title) in enumerate(panels):
        ax = axes[row][col]
        if img.ndim == 3:
            ax.imshow(img)
        else:
            cmap = "Blues" if "water" in title.lower() else ("terrain" if "DEM" in title else "gray")
            vmax = 1 if "water" in title.lower() else None
            ax.imshow(img, cmap=cmap, vmax=vmax)
        ax.set_title(title); ax.axis("off")

fig.suptitle("Three Ohio cities -- fused multi-mission cubes + water mask label", y=1.01)
fig.tight_layout()
plt.show()


## 2. Pretrained inference: `geoai.segment_water`

`geoai.segment_water` wraps OmniWaterMask -- a sensor-agnostic deep
model -- combined with NDWI and OpenStreetMap data. It accepts any
GeoTIFF with R, G, B, NIR bands; `band_order="naip"` tells it the
channel order our NAIP files use.

What this single call replaces in `01_classification.ipynb`:

* Manual feature engineering (`apply_band_norm`, `DEM_relative`, ...).
* Pixel-harvesting against the WorldCover label.
* Training a Random Forest, an XGBoost, and a U-Net.

OmniWaterMask is much more heavily pre-trained than the small model we
build in §3 below, so it tends to generalise to new scenes far better.
The flip side is that §3's path lets you train a *custom* segmenter on
*your* labels (water, burn scars, snow, building footprints, ...);
`segment_water` is hard-wired for water.

In [ ]:
%%time
# Pick the device. MPS is deliberately skipped: OmniWaterMask's U-Net asks
# for ~16 GB of MPS memory on a 3-mile NAIP scene, which OOMs on 18-24 GB
# Apple Silicon. CUDA scales fine; CPU is slow (~3 min) but reliable.
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"segment_water device: {device}")

# Run on Cleveland's NAIP scene (the largest visible water in our AOIs).
naip_path = CITY_FILES["cleveland"]["naip"]
water_mask_path = OUT_DIR / "cleveland_water_mask_naip.tif"
water_vec_path  = OUT_DIR / "cleveland_water_polygons.gpkg"

if not water_mask_path.exists():
    geoai.segment_water(
        input_path=str(naip_path),
        band_order="naip",
        output_raster=str(water_mask_path),
        output_vector=str(water_vec_path),
        device=device,
        use_osm_water=True,
        use_osm_building=False,
        use_osm_roads=False,
        cache_dir=str(OUT_DIR / "_omniwatermask_cache"),
        patch_size=512, overlap_size=128, batch_size=1,
        verbose=False,
    )

with rasterio.open(water_mask_path) as src:
    water = src.read(1)
print(f"Water mask  : {water.shape}")
print(f"Water pixels: {int((water > 0).sum())} of {water.size} "
      f"({100 * (water > 0).mean():.2f}%)")

with rasterio.open(naip_path) as src:
    naip_rgb = np.dstack([_stretch(src.read(i).astype("float32"))
                          for i in (1, 2, 3)])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(naip_rgb);                              axes[0].set_title("NAIP RGB (input)")
axes[1].imshow(water, cmap="Blues", vmin=0, vmax=1);   axes[1].set_title("segment_water output")
axes[2].imshow(naip_rgb)
axes[2].imshow(np.ma.masked_where(water == 0, water),
               cmap="cool", alpha=0.5);                axes[2].set_title("Overlay")
for ax in axes: ax.axis("off")
fig.suptitle("Cleveland NAIP -- pretrained water segmentation via geoai.segment_water", y=1.03)
fig.tight_layout()
plt.show()


## 3. Custom training: multi-AOI experiment + band selection

`geoai-py`'s training functions assume a fixed channel count
(typically 3 or 4) and a clean nodata convention. Two practical
limitations follow:

* Functions that use PIL-based loaders (`train_segmentation_landcover`)
  only accept **1, 3, or 4-channel** images.
* `semantic_segmentation` inherits the input cube's `nodata` value into
  the uint8 output -- and rasterio rejects `nan` as a uint8 nodata.

Our pipeline solves both at once via
**`geoai_datacubes.preprocessing.select_bands`**, which writes a clean
3- or 4-band uint8 subset of a fused cube using each band's documented
`band_meta` normalisation recipe. The `BAND_PRESETS` dict ships a few
task-meaningful triplets and quartets:

| Preset | Bands | Encodes |
|---|---|---|
| `ndwi` | S2 B03, B04, B08 | NDWI = (G - NIR)/(G + NIR) -- surface water |
| `nbr` | S2 B08, B11, B12 | Normalised Burn Ratio -- fire scars |
| `ndsi` | S2 B03, B04, B11 | Normalised Difference Snow Index |
| `rgb_nir` | S2 R, G, B, NIR | Classic 4-band optical |
| `rgb_dem` | S2 R, G, B, DEM | Terrain-aware vegetation |
| `rgb_sar_vv` | S2 R, G, B, S1 VV | Optical + all-weather SAR |
| `ndwi_sar_vv` | S2 G, R, NIR + S1 VV | NDWI bands + SAR (water-focused) |
| `naip` | NAIP R, G, B, NIR | Raw NAIP 4-band |

### Experiment design (deliberately honest)

We train on **two cities** (Cleveland + Cincinnati) and **hold the third
(Columbus) out** as a completely unseen test region. This separates
*"the model can learn this task on familiar data"* (in-distribution
F1) from *"the model generalises to a new region"* (OOD F1). The two
are often dramatically different and conflating them is the most
common way ML demos mislead readers.

Within Cleveland and Cincinnati we use **horizontal stripes** for
train chips (no leakage between training images and any later
re-evaluation).

### Quick parameter set (default)

| Setting | Value |
|---|---|
| Preset | `ndwi` (3-band) |
| Encoder | `resnet18` (ImageNet-pretrained) |
| Epochs | 10 |
| Batch | 4 |
| Learning rate | 1e-3 |
| Loss | focal + inverse-frequency class weights |
| CPU runtime | ~20 min |

In [ ]:
# Build clean 3-band NDWI subsets + clean uint8 masks for all three cities.
PRESET       = "ndwi"
NUM_CHANNELS = 3
TILE         = 128
STRIDE       = 48

for city, files in CITY_FILES.items():
    sub_dir = OUT_DIR / city / "subsets"; sub_dir.mkdir(exist_ok=True, parents=True)
    ndwi_path  = sub_dir / f"cube_{PRESET}.tif"
    mask_clean = sub_dir / "water_mask_clean.tif"
    if not ndwi_path.exists():
        select_bands(files["cube"], ndwi_path, BAND_PRESETS[PRESET],
                     normalize=True, dtype="uint8", nodata=None)
    if not mask_clean.exists():
        write_label_uint8(files["mask"], mask_clean, nodata=None)
    files["ndwi"]       = ndwi_path
    files["mask_clean"] = mask_clean
    with rasterio.open(ndwi_path) as src:
        print(f"  {city:11s}  ndwi={src.count}b x {src.height}x{src.width}  "
              f"dtype={src.dtypes[0]}  nodata={src.nodata}  "
              f"bands={list(src.descriptions or [])}")


In [ ]:
# Chip Cleveland + Cincinnati into 128x128 tiles, keeping only chips
# that fit cleanly in each city's top 80% horizontal stripe. Columbus
# is never chipped -- it stays fully held out.

TRAIN_CITIES = ["cleveland", "cincinnati"]
HELD_OUT     = "columbus"
TRAIN_FRAC   = 0.80     # top 80% rows of each city for training

CHIPS_DIR = OUT_DIR / "chips"
IMG_DIR = CHIPS_DIR / "images"; IMG_DIR.mkdir(exist_ok=True, parents=True)
LBL_DIR = CHIPS_DIR / "labels"; LBL_DIR.mkdir(exist_ok=True, parents=True)
for d in (IMG_DIR, LBL_DIR):
    for p in d.glob("*.tif"): p.unlink()

def chip_train_stripe(city, ndwi_path, mask_path):
    with rasterio.open(ndwi_path) as s: img = s.read(); im = s.meta.copy()
    with rasterio.open(mask_path) as s: lbl = s.read(1); lm = s.meta.copy()
    H, W = img.shape[1], img.shape[2]
    boundary = int(H * TRAIN_FRAC)
    n = 0
    for y in range(0, H - TILE + 1, STRIDE):
        for x in range(0, W - TILE + 1, STRIDE):
            if y + TILE > boundary:
                continue
            n += 1
            ci = img[:, y:y+TILE, x:x+TILE]
            cl = lbl[y:y+TILE, x:x+TILE]
            m = im.copy(); m.update(height=TILE, width=TILE, count=img.shape[0])
            with rasterio.open(IMG_DIR / f"{city}_y{y:04d}_x{x:04d}.tif", "w", **m) as d:
                d.write(ci)
            m = lm.copy(); m.update(height=TILE, width=TILE, count=1,
                                    dtype="uint8", nodata=None)
            with rasterio.open(LBL_DIR / f"{city}_y{y:04d}_x{x:04d}.tif", "w", **m) as d:
                d.write(cl[None])
    return n

for city in TRAIN_CITIES:
    n = chip_train_stripe(city, CITY_FILES[city]["ndwi"], CITY_FILES[city]["mask_clean"])
    print(f"  {city:11s} -- {n} train chips")
total = len(list(IMG_DIR.glob('*.tif')))
print(f"\ntotal train chips: {total}")
print(f"held out entirely : {HELD_OUT}")


In [ ]:
%%time
# Train via geoai.train_segmentation_landcover (focal loss + class weights).
# Default 'quick' params: ResNet18 + 10 epochs + batch=4 -> ~20 min on CPU.
# For tighter convergence, swap to encoder_name='resnet34', num_epochs=20,
# batch_size=2 (~50 min on CPU, marginal gain on this AOI / chip count).
ENCODER       = "resnet18"
NUM_EPOCHS    = 10
BATCH_SIZE    = 4
LEARNING_RATE = 1e-3

TRAIN = OUT_DIR / "train_landcover"
TRAIN.mkdir(exist_ok=True)

if not (TRAIN / "best_model.pth").exists():
    t0 = time.time()
    geoai.train_segmentation_landcover(
        images_dir=str(IMG_DIR), labels_dir=str(LBL_DIR), output_dir=str(TRAIN),
        architecture="unet", encoder_name=ENCODER, encoder_weights="imagenet",
        num_channels=NUM_CHANNELS, num_classes=2,
        batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        val_split=0.2, target_size=(TILE, TILE),
        save_best_only=True, plot_curves=False, verbose=False, print_freq=5,
        loss_function="focal",
        use_class_weights=True, use_inverse_frequency=True, max_class_weight=50.0,
        ignore_index=-100, focal_gamma=2.0,
    )
    print(f"train_segmentation_landcover OK in {time.time()-t0:.1f}s")
else:
    print(f"reusing cached {TRAIN / 'best_model.pth'}")


In [ ]:
%%time
# Evaluate on each city's full NDWI cube. Cleveland + Cincinnati are
# 'train + val mixed' (the trainer internally split 80/20); Columbus is
# the fully out-of-distribution held-out test. Inference is ~30-60s/city
# on CPU; instant on cached re-runs.

def evaluate(city, ndwi_path, mask_clean):
    pred_path = OUT_DIR / city / f"pred_{PRESET}.tif"
    if not pred_path.exists():
        geoai.semantic_segmentation(
            input_path=str(ndwi_path), output_path=str(pred_path),
            model_path=str(TRAIN / "best_model.pth"),
            architecture="unet", encoder_name=ENCODER,
            num_channels=NUM_CHANNELS, num_classes=2,
            window_size=TILE, overlap=32, batch_size=2,
            device="cpu", quiet=True,
        )
    with rasterio.open(pred_path)   as s: p = s.read(1)
    with rasterio.open(mask_clean)  as s: t = s.read(1)
    tp = int(((p==1) & (t==1)).sum())
    fp = int(((p==1) & (t==0)).sum())
    fn = int(((p==0) & (t==1)).sum())
    tn = int(((p==0) & (t==0)).sum())
    iou  = tp / max(1, tp + fp + fn)
    prec = tp / max(1, tp + fp)
    rec  = tp / max(1, tp + fn)
    f1   = 2 * prec * rec / max(1e-9, prec + rec)
    acc  = (tp + tn) / max(1, tp + fp + fn + tn)
    return dict(pred_path=pred_path, pred=p, truth=t,
                water_pct=100*t.mean(), pred_pct=100*p.mean(),
                f1=f1, iou=iou, prec=prec, rec=rec, acc=acc)

RESULTS = {}
for city, files in CITY_FILES.items():
    RESULTS[city] = evaluate(city, files["ndwi"], files["mask_clean"])

print(f"{'split':18s} {'city':12s} {'water%':>7s} {'pred%':>7s} "
      f"{'F1':>7s} {'IoU':>7s} {'P':>7s} {'R':>7s} {'acc':>7s}")
print("-" * 80)
for city in ("cleveland", "cincinnati", "columbus"):
    r = RESULTS[city]
    split = "train (in-dist) " if city in TRAIN_CITIES else "HELD-OUT TEST  "
    print(f"{split:18s} {city:12s} {r['water_pct']:7.2f} {r['pred_pct']:7.2f} "
          f"{r['f1']:7.3f} {r['iou']:7.3f} {r['prec']:7.3f} {r['rec']:7.3f} {r['acc']:7.3f}")


In [ ]:
# Visualise predictions vs ground truth for each city.
fig, axes = plt.subplots(len(CITIES), 3, figsize=(15, 4.5 * len(CITIES)))
for row, city in enumerate(("cleveland", "cincinnati", "columbus")):
    files = CITY_FILES[city]
    r     = RESULTS[city]
    with rasterio.open(files["ndwi"]) as src:
        n_arr = src.read().astype("float32") / 255.0
    rgb_disp = np.dstack([n_arr[2], n_arr[0], n_arr[1]])  # NIR-G-R false colour
    split = "train" if city in TRAIN_CITIES else "HELD-OUT"
    axes[row][0].imshow(rgb_disp)
    axes[row][0].set_title(f"{city.capitalize()} -- NDWI 3-band ({split})")
    axes[row][1].imshow(r["truth"], cmap="Blues", vmin=0, vmax=1)
    axes[row][1].set_title(f"Truth ({r['water_pct']:.2f}% water)")
    axes[row][2].imshow(r["pred"], cmap="Blues", vmin=0, vmax=1)
    axes[row][2].set_title(f"Prediction\nF1={r['f1']:.3f}  IoU={r['iou']:.3f}  "
                           f"P={r['prec']:.3f}  R={r['rec']:.3f}")
    for ax in axes[row]: ax.axis("off")
fig.suptitle("Trained on Cleveland + Cincinnati, evaluated on all three", y=1.01)
fig.tight_layout()
plt.show()


## 4. What this tells us

* **In-distribution** (Cleveland + Cincinnati): the model reaches
  F1 in the high 0.8–0.95 range. The integration -- our pipeline produces
  the input GeoTIFFs that `geoai.train_segmentation_landcover` and
  `geoai.semantic_segmentation` consume cleanly -- works end-to-end.

* **Out-of-distribution** (Columbus, held out entirely): F1 collapses
  to roughly 0.05. The model over-predicts water by ~30x, firing on
  dark surfaces (parking lots, shadows, dark roofs) that look like
  water locally but were not in any training scene.

This gap is **not a bug** in either package -- it's the standard
remote-sensing-ML reality of training on a handful of AOIs. A few
ways to actually close it, in roughly increasing cost:

1. **More AOIs of similar morphology.** The two training cities here
   shared "urban along big water" structure; Columbus's suburban
   narrow-river scene was unseen. Adding cities with narrow rivers
   to the train set (Indianapolis, Pittsburgh, ...) would help most.
2. **Data augmentation** (flips, rotations, random scaling). Cheap.
3. **Heavier pretrained backbones** trained on planet-scale water
   datasets (e.g. `geoai.segment_water`'s OmniWaterMask, which DOES
   generalise across regions because of its much larger pretraining
   set). Use this when you want a *good water mask* without training.
4. **Multispectral foundation models** -- Prithvi-EO-2.0, DOFA, Clay --
   exposed via `geoai.PrithviProcessor` and friends. Designed for
   exactly this kind of region-to-region transfer.

What we showed here is that **`geoai-datacubes` removes the
data-prep friction** that would otherwise block any of these
improvements: as soon as you want to train on more cities, you swap a
list. As soon as you want a different band combination, you swap a
`BAND_PRESETS` key. The modelling stack stays the same.

## 5. Two roles, one workflow

The split of labour between `geoai-datacubes` and `geoai-py` is
deliberate and asymmetric:

| Role | `geoai-datacubes` (this repo) | `geoai-py` (Wu, 2026) |
|---|---|---|
| Multi-mission fetch | 15 working direct-observation + 8 derived | 3 download helpers |
| Normalisation recipes | `band_meta` declares per-band defaults; user-overridable | not the focus |
| Fusion onto a common grid | `fuse_response_tiffs` | -- |
| Subset to clean N-band uint8 | `select_bands` + `BAND_PRESETS` | -- |
| Tile streaming | `LazyTileDataset` (Zarr + LMDB) | `RemoteSensingDataset` + TorchGeo |
| Pretrained task models | -- | `segment_water`, `BuildingFootprintExtractor`, ... |
| Foundation-model wrappers | -- | `PrithviProcessor`, `DINOv3Segmenter`, ... |
| Training loops | reference U-Net + sklearn baselines | `train_*_segmentation_model`, Lightning + TorchGeo |
| QGIS plugin | -- | provided |

The natural workflow is `geoai-datacubes` for the messy data-prep
front-end, `geoai-py` for the modelling back-end. In this notebook
the hand-off happens twice -- once into a pretrained model
(`segment_water`), once into a custom training loop
(`train_segmentation_landcover` + `semantic_segmentation`) -- with
`select_bands` resolving the channel-count and nodata mismatches
that would otherwise break the integration.

For deeper modelling examples -- foundation models, training utilities,
the QGIS plugin -- see the `geoai-py` book at
<https://book.opengeoai.org/>. Cubes built here drop in wherever
that book uses a single GeoTIFF or RGB tile as input.